## **Assignment Simple machine Learning Project cycle**

### **_House Price Prediction Using California Housing Data_**
This dataset provides information about house prices in California. House Price Prediction


**In this notebook, you'll follow the basic machine learning process to build a regression model to predict house prices using the "California Housing Dataset" from sklearn.**



---



### **Follow the instructions and complete each TODO to complete the assessment on the essential steps in building and evaluating a regression model.**


The following is a description of each column in the dataset:

Dataset Features (California Housing):
* MedInc: Median income in block group
* HouseAge: Median house age in block group
* AveRooms: Average number of rooms per household
* AveBedrms: Average number of bedrooms per household
* Population: Block group population
* AveOccup: Average number of household members
* Latitude: Block group latitude
* Longitude: Block group longitude
* MedHouseVal (Target): Median house value in block group




In [ ]:
# ============================================================
# STEP 1 — IMPORTS
# Import all necessary libraries for data handling,
# visualization, and model building.
# ============================================================

import pandas as pd                         # Data manipulation
import numpy as np                          # Numerical computing
import matplotlib.pyplot as plt             # Plotting
import seaborn as sns                       # Statistical visualizations

from sklearn.datasets import fetch_california_housing          # Dataset
from sklearn.model_selection import train_test_split           # Data splitting
from sklearn.linear_model import LinearRegression              # Regression model
from sklearn.metrics import mean_squared_error, r2_score       # Evaluation metrics

# Set a consistent plot style
sns.set_theme(style='whitegrid')
print('All libraries imported successfully.')


In [ ]:
# ============================================================
# STEP 2 — DATA COLLECTION AND LOADING
# Load the California Housing dataset and convert it
# into a pandas DataFrame.
# ============================================================

# Load the dataset from sklearn
housing = fetch_california_housing()

# Build DataFrame from feature matrix
df = pd.DataFrame(housing.data, columns=housing.feature_names)

# Add the target variable (Median House Value) as a new column
df['MedHouseVal'] = housing.target

print('Dataset loaded successfully.')
print(f'Shape: {df.shape}  ({df.shape[0]} rows, {df.shape[1]} columns)')


In [ ]:
# ============================================================
# STEP 3 — QUICK CHECK OF DATA
# Inspect the structure, types, and basic statistics
# of the dataset.
# ============================================================

# 3a. First few rows
print('--- First 5 rows ---')
display(df.head())

# 3b. Data types and non-null counts
print('\n--- Dataset Info ---')
df.info()

# 3c. Descriptive statistics
print('\n--- Descriptive Statistics ---')
display(df.describe())

# 3d. Feature type analysis
print('\nFeature types:')
print('  Continuous (numerical): MedInc, HouseAge, AveRooms, AveBedrms,')
print('                          Population, AveOccup, Latitude, Longitude')
print('  Target (continuous):    MedHouseVal')
print('  Categorical:            None — all features are numerical.')


In [ ]:
# ============================================================
# STEP 4 — EDA AND DATA PREPROCESSING
# ============================================================

# 4a. Check for missing / null values
print('--- Missing Values per Column ---')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')
# No missing values in this dataset — no imputation needed.


In [ ]:
# 4b. Single scatter plot — MedInc vs MedHouseVal
# (quick visual check before building the automated function)

plt.figure(figsize=(7, 5))
plt.scatter(df['MedInc'], df['MedHouseVal'],
            alpha=0.3, color='steelblue', edgecolors='none', s=10)
plt.title('Median Income vs Median House Value', fontsize=14)
plt.xlabel('MedInc (Median Income)', fontsize=12)
plt.ylabel('MedHouseVal (Median House Value, $100k)', fontsize=12)
plt.tight_layout()
plt.show()
print('Observation: A clear positive correlation — higher income areas tend to have higher house values.')


In [ ]:
# 4c. Automated scatter-plot function

def plot_features_vs_target(dataframe, features, target='MedHouseVal'):
    """
    Plot scatter plots for each feature in `features` against the `target` column.

    Parameters
    ----------
    dataframe : pd.DataFrame
        The dataset containing all columns.
    features  : list of str
        Column names to plot on the x-axis.
    target    : str
        Column name for the y-axis (default: 'MedHouseVal').
    """
    n = len(features)
    n_cols = 2
    n_rows = (n + 1) // n_cols   # Compute rows needed

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
    axes = axes.flatten()        # Flatten for easy iteration

    for i, feature in enumerate(features):
        axes[i].scatter(dataframe[feature], dataframe[target],
                        alpha=0.3, s=10, color='teal', edgecolors='none')
        axes[i].set_title(f'{feature} vs {target}', fontsize=13)
        axes[i].set_xlabel(feature, fontsize=11)
        axes[i].set_ylabel(target, fontsize=11)

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle('Feature Relationships with Median House Value',
                 fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()


print('Function plot_features_vs_target() defined.')


In [ ]:
# 4d. Use the function to visualize key features

selected_features = ['MedInc', 'AveRooms', 'AveOccup', 'HouseAge']

plot_features_vs_target(df, features=selected_features, target='MedHouseVal')

print("""
Observations:
  - MedInc    : Strong positive correlation with house value — most important predictor.
  - AveRooms  : Slight positive trend; outliers with very large room counts distort the pattern.
  - AveOccup  : Mostly clustered at low values; extreme outliers suggest data skew.
  - HouseAge  : No strong linear relationship; older and newer homes span similar price ranges.
""")


In [ ]:
# ============================================================
# STEP 5 — ML MODEL TRAINING
# ============================================================

# 5a. Define features (X) and target (y)
X = df.drop(columns=['MedHouseVal'])   # All 8 input features
y = df['MedHouseVal']                  # Target: median house value ($100k)

print(f'Feature matrix shape : {X.shape}')
print(f'Target vector shape  : {y.shape}')

# 5b. Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print(f'\nTraining set size : {X_train.shape[0]} samples')
print(f'Testing set size  : {X_test.shape[0]} samples')

# 5c. Instantiate the Linear Regression model
# LinearRegression is appropriate for continuous target prediction
# and serves as a strong, interpretable baseline.
model = LinearRegression()

# 5d. Train the model on the training data
model.fit(X_train, y_train)

print('\nModel trained successfully.')
print('Learned coefficients (one per feature):')
for name, coef in zip(X.columns, model.coef_):
    print(f'  {name:12s}: {coef:+.4f}')
print(f'  Intercept   : {model.intercept_:+.4f}')


In [ ]:
# ============================================================
# STEP 6 — MODEL EVALUATION
# ============================================================

# 6a. Generate predictions on the test set
y_pred = model.predict(X_test)

# 6b. Calculate evaluation metrics
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print('--- Model Evaluation on Test Set ---')
print(f'  MSE  (Mean Squared Error)    : {mse:.4f}')
print(f'  RMSE (Root Mean Sq. Error)   : {rmse:.4f}  (in $100k units)')
print(f'  R²   (Coefficient of Det.)   : {r2:.4f}')

print("""
Interpretation:
  - RMSE ≈ 0.73 means the model's predictions are off by ~$73,000 on average.
  - R² ≈ 0.60 means the model explains ~60% of the variance in house prices.
  - Linear Regression provides a useful baseline but a non-linear model
    (e.g. Random Forest, Gradient Boosting) would likely improve performance.
""")

# 6c. Residual plot — actual vs predicted
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, alpha=0.3, s=10, color='darkorange')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', linewidth=1.5, label='Perfect prediction')
plt.xlabel('Actual MedHouseVal ($100k)', fontsize=12)
plt.ylabel('Predicted MedHouseVal ($100k)', fontsize=12)
plt.title('Actual vs Predicted Median House Value', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# STEP 7 — MODEL PREDICTION
# Predict the median house value for a new hypothetical
# block group with the following characteristics:
#   MedInc=3, HouseAge=30, AveRooms=6, AveBedrms=1,
#   Population=1500, AveOccup=3, Latitude=34, Longitude=-118
# ============================================================

# Build the new data point in the same column order as X
new_data = pd.DataFrame([{
    'MedInc'    : 3.0,
    'HouseAge'  : 30.0,
    'AveRooms'  : 6.0,
    'AveBedrms' : 1.0,
    'Population': 1500.0,
    'AveOccup'  : 3.0,
    'Latitude'  : 34.0,
    'Longitude' : -118.0
}])

# Make the prediction
predicted_value = model.predict(new_data)[0]

print('--- New Prediction ---')
print('Input features:')
print(new_data.to_string(index=False))
print(f'\nPredicted Median House Value: ${predicted_value:.4f} (×$100,000)')
print(f'  ≈ ${predicted_value * 100_000:,.0f}')
print("""
Interpretation:
  For a block group in the Los Angeles area (Lat 34, Long -118) with a
  median income of $30,000, an average house age of 30 years, 6 rooms,
  1 bedroom, and a population of 1500, the model predicts a median house
  value of approximately the amount shown above.
""")
